In [ ]:
# Setup: Import required libraries and configure plotting
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set seaborn style to whitegrid and default figure size to (10, 6)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# Load the processed dataset with datetime parsing
data_path = "../data/processed_youtube_data.csv"
df = pd.read_csv(data_path, parse_dates=["trending_date", "publish_time"])
print(f"Successfully loaded {len(df):,} records from {data_path}")

# YouTube Trending Video Analysis - Exploratory Data Analysis

This notebook performs exploratory data analysis (EDA) on trending YouTube videos across multiple regions (US, GB, IN). We inspect data characteristics, compute engagement and ratio metrics, and evaluate performance trends across different video categories.

In [ ]:
# Basic Overview: Dataset shape, structure, and numerical summary
print(f"Dataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns\n")

print("=" * 50)
print("DATASET INFO")
print("=" * 50)
df.info()

print("\n" + "=" * 50)
print("SUMMARY STATISTICS FOR NUMERIC COLUMNS")
print("=" * 50)
numeric_cols = ["views", "likes", "dislikes", "comment_count"]
df[numeric_cols].describe()

In [ ]:
# Engagement Metrics: Calculate interaction rates and like ratios
# 1. engagement_rate: total user interactions relative to views
df["engagement_rate"] = (df["likes"] + df["dislikes"] + df["comment_count"]) / df["views"]

# 2. like_ratio: positive sentiment fraction (handles division by zero when reactions = 0)
total_reactions = df["likes"] + df["dislikes"]
df["like_ratio"] = np.where(total_reactions > 0, df["likes"] / total_reactions, 0.0)

# 3. Top 10 videos by engagement rate
top_10_engagement = df.sort_values(by="engagement_rate", ascending=False)[
    ["title", "country", "views", "engagement_rate"]
].head(10)

print("Top 10 Videos by Engagement Rate:")
top_10_engagement

## Category-wise Analysis

Here we aggregate the dataset by `category_name` to compare average views, engagement rates, and total video counts across different genres.

In [ ]:
# Category Analysis: Aggregate metrics by category_name
category_summary = (
    df.groupby("category_name")
    .agg(
        mean_views=("views", "mean"),
        mean_engagement_rate=("engagement_rate", "mean"),
        video_count=("video_id", "count"),
    )
    .sort_values(by="mean_views", ascending=False)
    .reset_index()
)

# Display summary DataFrame
print("Category Performance Summary (sorted by mean views descending):")
display(category_summary)

# Filter Top 10 categories by average views
top10_categories = category_summary.head(10).sort_values(by="mean_views", ascending=True)

# Horizontal bar chart
plt.figure(figsize=(10, 6))
palette = sns.color_palette("Blues_r", n_colors=len(top10_categories))

bars = plt.barh(
    top10_categories["category_name"],
    top10_categories["mean_views"] / 1e6,
    color=palette[::-1],
    edgecolor="none",
    height=0.65,
)

plt.title("Top 10 YouTube Categories by Average Views", fontsize=15, fontweight="bold", pad=15)
plt.xlabel("Average Views (Millions)", fontsize=12, labelpad=10)
plt.ylabel("Category", fontsize=12, labelpad=10)

# Annotate bars with data values
max_val = (top10_categories["mean_views"] / 1e6).max()
for bar in bars:
    width = bar.get_width()
    plt.text(
        width + (max_val * 0.015),
        bar.get_y() + bar.get_height() / 2,
        f"{width:.2f}M",
        va="center",
        ha="left",
        fontsize=10,
        fontweight="semibold",
        color="#2c3e50",
    )

plt.xlim(0, max_val * 1.15)
plt.tight_layout()
plt.savefig("../outputs/category_avg_views.png", dpi=300, bbox_inches="tight")
plt.show()

## Time-Based Publishing Trends

Analyzing the timing of video publication provides actionable insights into content release strategies. Below, we examine trends across hours of the day (0–23 UTC) and days of the week to identify peak uploading frequency and maximum average viewership windows.

In [ ]:
# Publishing Hour Analysis: Aggregate video count and average views by hour (0-23)
hourly_trends = (
    df.groupby("publish_hour")
    .agg(
        video_count=("video_id", "count"),
        mean_views=("views", "mean"),
    )
    .reset_index()
)

# Create combined 2-subplot layout for publishing hour trends
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 9))

# 1. Line plot: Number of videos published per hour
ax1.plot(
    hourly_trends["publish_hour"],
    hourly_trends["video_count"],
    marker="o",
    color="#1f77b4",
    linewidth=2.5,
    markersize=6,
    label="Video Upload Count",
)
ax1.fill_between(hourly_trends["publish_hour"], hourly_trends["video_count"], color="#1f77b4", alpha=0.15)
ax1.set_title("Volume of Videos Published by Hour of Day (0-23)", fontsize=14, fontweight="bold", pad=12)
ax1.set_xlabel("Hour of Day (UTC)", fontsize=12, labelpad=8)
ax1.set_ylabel("Number of Videos Published", fontsize=12, labelpad=8)
ax1.set_xticks(range(0, 24))
ax1.grid(True, linestyle="--", alpha=0.6)

# 2. Bar plot: Average views by publish hour
ax2.bar(
    hourly_trends["publish_hour"],
    hourly_trends["mean_views"] / 1e6,
    color="#2b5c8f",
    width=0.7,
    edgecolor="none",
)
ax2.set_title("Average Views by Publishing Hour", fontsize=14, fontweight="bold", pad=12)
ax2.set_xlabel("Hour of Day (UTC)", fontsize=12, labelpad=8)
ax2.set_ylabel("Average Views (Millions)", fontsize=12, labelpad=8)
ax2.set_xticks(range(0, 24))
ax2.grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.savefig("../outputs/publish_hour_trends.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# Day of Week Analysis: Reindex days chronologically from Monday to Sunday
days_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
dow_summary = (
    df.groupby("publish_day_of_week")
    .agg(
        mean_views=("views", "mean"),
        video_count=("video_id", "count"),
    )
    .reindex(days_order)
    .reset_index()
)

# Identify the day with the highest average views
max_views_day = dow_summary.loc[dow_summary["mean_views"].idxmax(), "publish_day_of_week"]
print(f"Day with highest average views: {max_views_day} ({dow_summary['mean_views'].max():,.0f} views)")

# Define bar colors: highlight the top day in vibrant red, rest in blue
bar_colors = [
    "#e74c3c" if day == max_views_day else "#3498db"
    for day in dow_summary["publish_day_of_week"]
]

# Create bar chart
plt.figure(figsize=(10, 6))
bars = plt.bar(
    dow_summary["publish_day_of_week"],
    dow_summary["mean_views"] / 1e6,
    color=bar_colors,
    width=0.6,
)

# Annotate values above each bar
for bar in bars:
    height = bar.get_height()
    plt.annotate(
        f"{height:.2f}M",
        xy=(bar.get_x() + bar.get_width() / 2, height),
        xytext=(0, 4),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="semibold",
    )

plt.title("Average Views by Day of Week (Peak Day Highlighted)", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Day of Week", fontsize=12, labelpad=10)
plt.ylabel("Average Views (Millions)", fontsize=12, labelpad=10)
plt.ylim(0, (dow_summary["mean_views"].max() / 1e6) * 1.15)
plt.tight_layout()
plt.savefig("../outputs/day_of_week_views.png", dpi=300, bbox_inches="tight")
plt.show()

## Country-wise Comparison

Comparing key viewership metrics across the United States (US), Great Britain (GB), and India (IN) reveals notable contrasts in average audience reach and engagement enthusiasm.

In [ ]:
# Country Comparison: Group by country and calculate mean views, mean likes, mean engagement_rate, and total count
country_summary = (
    df.groupby("country")
    .agg(
        mean_views=("views", "mean"),
        mean_likes=("likes", "mean"),
        mean_engagement_rate=("engagement_rate", "mean"),
        video_count=("video_id", "count"),
    )
    .reset_index()
)

print("Country-wise Metric Summary:")
display(country_summary)

# Grouped bar chart with dual y-axes for Mean Views and Mean Engagement Rate
x = np.arange(len(country_summary))
width = 0.35

fig, ax1 = plt.subplots(figsize=(10, 6))
ax2 = ax1.twinx()

# Primary axis: Mean Views (in Millions)
bars1 = ax1.bar(
    x - width / 2,
    country_summary["mean_views"] / 1e6,
    width=width,
    color="#2b5c8f",
    label="Mean Views (Millions)",
)

# Secondary axis: Mean Engagement Rate (in %)
bars2 = ax2.bar(
    x + width / 2,
    country_summary["mean_engagement_rate"] * 100,
    width=width,
    color="#e67e22",
    label="Mean Engagement Rate (%)",
)

# Axis labels and formatting
ax1.set_title("Country Comparison: Mean Views vs. Mean Engagement Rate", fontsize=14, fontweight="bold", pad=15)
ax1.set_xlabel("Country", fontsize=12, labelpad=10)
ax1.set_ylabel("Mean Views (Millions)", color="#2b5c8f", fontsize=12, labelpad=8)
ax2.set_ylabel("Mean Engagement Rate (%)", color="#e67e22", fontsize=12, labelpad=8)
ax1.set_xticks(x)
ax1.set_xticklabels(country_summary["country"], fontsize=11, fontweight="bold")

# Gridlines only on primary axis to avoid clutter
ax1.grid(True, linestyle="--", alpha=0.5)
ax2.grid(False)

# Annotate bar values
for bar in bars1:
    ax1.annotate(
        f"{bar.get_height():.2f}M",
        xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
        xytext=(0, 4),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="semibold",
        color="#2b5c8f",
    )
for bar in bars2:
    ax2.annotate(
        f"{bar.get_height():.2f}%",
        xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
        xytext=(0, 4),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="semibold",
        color="#e67e22",
    )

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right", frameon=True)

plt.tight_layout()
plt.savefig("../outputs/country_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

## Category Preferences by Country

Content consumption behaviors differ markedly between regions. Here, we build a cross-country pivot table of video counts and visualize the top 10 categories using an annotated heatmap.

In [ ]:
# Category Preferences by Country: Pivot table of video counts
category_pivot = df.pivot_table(
    index="category_name",
    columns="country",
    values="video_id",
    aggfunc="count",
    fill_value=0,
)

# Calculate total across countries and filter top 10 categories
category_pivot["Total"] = category_pivot.sum(axis=1)
top10_categories_pivot = category_pivot.sort_values(by="Total", ascending=False).head(10)

print("Top 10 Categories by Total Video Count Across Countries:")
display(top10_categories_pivot)

# Heatmap of Top 10 Categories across countries (excluding Total column)
heatmap_data = top10_categories_pivot.drop(columns=["Total"])

plt.figure(figsize=(10, 8))
sns.heatmap(
    heatmap_data,
    annot=True,
    fmt="d",
    cmap="YlGnBu",
    linewidths=0.75,
    linecolor="#ffffff",
    cbar_kws={"label": "Trending Video Count"},
)

plt.title("Top 10 Video Categories by Country (Video Counts)", fontsize=15, fontweight="bold", pad=15)
plt.xlabel("Country", fontsize=12, labelpad=10)
plt.ylabel("Category", fontsize=12, labelpad=10)
plt.tight_layout()
plt.savefig("../outputs/category_country_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()